# 04 · Carbon Intensity Join & Correlation — Phase 4

**Research questions:**
- How much CO₂ is emitted by idle (unallocated) GPU capacity?
- Is the allocation gap correlated with carbon intensity (enabling carbon-aware scheduling)?
- How much CO₂ could deferrable jobs save by shifting to low-carbon windows?

**Prerequisites:** `make preprocess` completed for v2026.

In [ ]:
import sys
print(sys.executable)

import json
import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from IPython.display import display

from src.config import (
    GRID_PROFILES, CALENDAR_ANCHORS, FIG_DIR, TABLE_DIR,
    DUCKDB_MEMORY_LIMIT, DUCKDB_THREADS,
)
from src.viz.style import set_ieee_style
import src.analysis.carbon_fetch as cf
import src.analysis.carbon_join as cj
import src.analysis.correlation as corrmod

set_ieee_style()
FIG_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

con = duckdb.connect()
con.execute(f"SET memory_limit='{DUCKDB_MEMORY_LIMIT}'")
con.execute(f"SET threads={DUCKDB_THREADS}")
print('Ready')

> ⚠️ **Carbon intensities are synthetic estimates.**  
> Replace with Electricity Maps historical data when the API key arrives (Phase 4.5).

## Cell 1 — Synthetic carbon profile: one-week visualisation

In [ ]:
ANCHOR = '2025-01-01'
WEEK_H = 7 * 24

fig, ax = plt.subplots(figsize=(7.0, 2.8))
colors = ['#0072B2', '#E69F00', '#009E73', '#CC79A7']

for (region, _), color in zip(GRID_PROFILES.items(), colors):
    cp = cf.get_carbon_profile(region, ANCHOR)
    week = cp[cp['rel_hour'] < WEEK_H]
    ax.plot(week['rel_hour'], week['carbon_intensity_gco2_per_kwh'],
            label=region, color=color, lw=1.0)

for d in range(1, 8):
    ax.axvline(d * 24, color='grey', lw=0.3, ls=':')

ax.set_xlabel('Trace hour (day 0–6)')
ax.set_ylabel('Carbon intensity\n(gCO₂/kWh)')
ax.set_title('Synthetic carbon profiles — one week (anchor: 2025-01-01)')
ax.legend(loc='upper right', handlelength=1.2, fontsize=6)
ax.text(0.01, 0.97,
        '⚠️ Synthetic profiles; real Electricity Maps data pending API key approval',
        transform=ax.transAxes, fontsize=5, color='#AA4400',
        va='top', style='italic')

plt.tight_layout()
plt.savefig(FIG_DIR / 'fig5_synthetic_carbon_profiles.pdf')
plt.show()
print('Saved: fig5_synthetic_carbon_profiles.pdf')

## Cell 2 — Headline table: idle CO₂ (tonnes) by anchor × region

> ⚠️ Carbon intensities are synthetic estimates.

In [ ]:
print('Computing 3×4 headline table (~30s) …')
ht = cj.headline_table(con)
print('\n=== Idle CO₂ (tonnes) by anchor × region ===')
display(
    ht.style
      .format('{:,.1f}')
      .background_gradient(cmap='YlOrRd', axis=None)
      .set_caption('⚠️ Synthetic carbon profiles — illustrative only')
)
ht.to_csv(TABLE_DIR / 'table2_idle_co2_by_anchor_region.csv')
print(f'Saved: {TABLE_DIR}/table2_idle_co2_by_anchor_region.csv')

## Cell 3 — Figure 6: dual-axis time series (one representative week)

> ⚠️ Carbon intensities are synthetic estimates.

In [ ]:
ANCHOR_FIG = '2025-01-01'
WEEK_H = 7 * 24

regions_list = list(GRID_PROFILES.keys())
fig, axes = plt.subplots(2, 2, figsize=(7.0, 4.5))
axes_flat = axes.flatten()

for ax, region in zip(axes_flat, regions_list):
    hc = cj.compute_hourly_carbon(region, ANCHOR_FIG, con=con)
    week = hc[hc['rel_hour'] < WEEK_H]

    ax2 = ax.twinx()
    ax.fill_between(week['rel_hour'], week['idle_gpus'] / 1000,
                    alpha=0.3, color='#0072B2', label='Idle GPUs (K)')
    ax.plot(week['rel_hour'], week['idle_gpus'] / 1000,
            color='#0072B2', lw=0.8)
    ax2.plot(week['rel_hour'], week['carbon_intensity_gco2_per_kwh'],
             color='#D55E00', lw=1.0, ls='--', label='Carbon (gCO₂/kWh)')

    for d in range(1, 8):
        ax.axvline(d * 24, color='grey', lw=0.3, ls=':')

    ax.set_xlabel('Trace hour')
    ax.set_ylabel('Idle GPUs (K)', color='#0072B2')
    ax2.set_ylabel('gCO₂/kWh', color='#D55E00')
    ax.set_title(region)

plt.suptitle(
    'Allocation gap vs. carbon intensity — representative week\n'
    '⚠️ Synthetic carbon; anchor: 2025-01-01',
    fontsize=7, style='italic'
)
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig6_gap_vs_carbon_dual_axis.pdf')
plt.show()
print('Saved: fig6_gap_vs_carbon_dual_axis.pdf')

## Cell 4 — Figure 7: correlation heatmap (Spearman ρ at each lag)

> ⚠️ Carbon intensities are synthetic estimates. Diurnal correlations driven by model structure.

In [ ]:
print('Running correlation analysis (500-iter block bootstrap, ~60s) …')
corr_df = corrmod.run_correlation_analysis(con)

# Median across anchors (calendar shift is a sensitivity, not signal)
lag0 = corr_df.groupby(['region', 'lag_hours'])['spearman_r'].median().reset_index()
pvals = corr_df.groupby(['region', 'lag_hours'])['p_value'].median().reset_index()

pivot   = lag0.pivot(index='region', columns='lag_hours', values='spearman_r')
piv_p   = pvals.pivot(index='region', columns='lag_hours', values='p_value')

fig, ax = plt.subplots(figsize=(7.0, 2.4))
sns.heatmap(
    pivot, ax=ax, cmap='RdBu_r', center=0, vmin=-0.5, vmax=0.5,
    annot=pivot.round(2), fmt='.2f', annot_kws={'size': 6},
    cbar_kws={'label': 'Spearman ρ', 'shrink': 0.8},
    linewidths=0.3,
)
for i, region in enumerate(pivot.index):
    for j, lag in enumerate(pivot.columns):
        p = piv_p.loc[region, lag]
        if not pd.isna(p) and p < 0.05:
            ax.text(j + 0.85, i + 0.25, '*', fontsize=8, color='black', ha='center')

ax.set_title(
    'Spearman ρ: allocation gap vs. carbon intensity by lag\n'
    '(* = p < 0.05  |  ⚠️ synthetic carbon profiles)',
    fontsize=7
)
ax.set_xlabel('Lag (hours)')
ax.set_ylabel('')
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig7_correlation_heatmap.pdf')
plt.show()
print('Saved: fig7_correlation_heatmap.pdf')

## Cell 5 — Deferrable savings table

> ⚠️ Carbon intensities are synthetic estimates — upper-bound figures only.

In [ ]:
sav = cj.compute_deferrable_savings()

by_type_region = (
    sav.groupby(['region', 'job_type_public'])
       .agg(
           total_deferrable_gpu_hours=('deferrable_gpu_hours', 'sum'),
           mean_carbon_saving_pct=('carbon_saving_pct', 'mean'),
           total_co2_saving_kg_ub=('co2_saving_kg_upper_bound', 'sum'),
       )
       .round({'total_deferrable_gpu_hours': 0,
               'mean_carbon_saving_pct': 2,
               'total_co2_saving_kg_ub': 0})
       .reset_index()
)

print('=== Deferrable CO₂ savings by region × job type ===')
print('(⚠️ Synthetic carbon — upper-bound estimates)')
display(by_type_region)
by_type_region.to_csv(TABLE_DIR / 'table3_deferrable_savings.csv', index=False)
print(f'Saved: {TABLE_DIR}/table3_deferrable_savings.csv')

## Cell 6 — Write outputs/tables/phase4_headline.json

In [ ]:
# ⚠️ Carbon intensities are synthetic estimates.
from pathlib import Path

ht_df = cj.headline_table(con)
sav_df = cj.compute_deferrable_savings()
corr_lag0 = corr_df[corr_df['lag_hours'] == 0]

# Idle CO₂ by region × anchor
total_idle = {}
for region in ht_df.columns:
    total_idle[region] = {anchor: float(ht_df.loc[anchor, region])
                          for anchor in ht_df.index}

# Deferrable savings by region
defer_by_region = (
    sav_df.groupby('region')
          .agg(saving_kg=('co2_saving_kg_upper_bound', 'sum'),
               baseline_kg=('co2_baseline_kg', 'sum'))
)
defer_co2_saving = {r: round(float(row['saving_kg'] / 1000), 1)
                    for r, row in defer_by_region.iterrows()}
defer_saving_pct = {r: round(float(row['saving_kg'] / row['baseline_kg'] * 100), 2)
                    for r, row in defer_by_region.iterrows()}

# Best correlation (lag = 0, median across anchors)
best_idx = corr_lag0.groupby('region')['spearman_r'].median().abs().idxmax()
best_r   = float(corr_lag0.groupby('region')['spearman_r'].median()[best_idx])

headline = {
    'total_idle_co2_tonnes': total_idle,
    'deferrable_co2_saving_upper_bound_tonnes': defer_co2_saving,
    'deferrable_saving_pct_of_total': defer_saving_pct,
    'best_correlation_region': best_idx,
    'best_correlation_coeff': round(best_r, 4),
    'note': 'synthetic carbon profiles — pending real Electricity Maps API data',
}

out_json = TABLE_DIR / 'phase4_headline.json'
with open(out_json, 'w') as f:
    json.dump(headline, f, indent=2)

print('=== Phase 4 headline JSON ===')
print(json.dumps(headline, indent=2))
print(f'\nSaved: {out_json}')